In [ ]:
# Path config (auto-injected). Loads WORK_DIR from .env so notebooks run anywhere.
import os
from pathlib import Path
from dotenv import load_dotenv

_p = Path.cwd().resolve()
for _ in range(8):
    if (_p / '.env').is_file():
        load_dotenv(_p / '.env')
        break
    if _p == _p.parent:
        break
    _p = _p.parent

WORK_DIR = os.environ['WORK_DIR']
RETRIEVAL_DIR = f"{WORK_DIR}/retrieval_assets"

# LLM Labeling for Natural Hallucination Test

This notebook labels the JSON produced by `12-natural-hallucination-test.ipynb` using an LLM judge.

Label convention:
- `0` = fully supported / correct
- `1` = hallucinated / unsupported / contradicted

The judge uses the full retrieved context, question, gold answer, and generated answer. It saves both labeled JSON and CSV outputs, then evaluates the metrics already stored in the input JSON.

## Step 1: Install and Configure

In [ ]:
!pip install -q groq pandas tqdm scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip install groq

In [ ]:
print("API KEY:", os.environ.get("GROQ_API_KEY"))
GROQ_API_KEY = os.environ['GROQ_API_KEY']  # Replace with actual key

In [ ]:
import os
os.environ["GROQ_API_KEY"] = os.environ['GROQ_API_KEY']

In [ ]:
import os
import json
import time
import re
import pandas as pd
import numpy as np
from tqdm import tqdm
from groq import Groq
from sklearn.metrics import roc_auc_score

# Update this path to the JSON created by 12-natural-hallucination-test.ipynb
INPUT_JSON = f"{WORK_DIR}/llama 8b/hotpot/tests/12/natural_hallucination_test_k1.json"

# Output files
OUTPUT_JSON = "natural_hallucination_test_k1_LLM_LABELED.json"
OUTPUT_CSV = "natural_hallucination_test_k1_LLM_LABELED.csv"
CHECKPOINT_JSON = "natural_hallucination_llm_label_checkpoint.json"

# Put your key in Colab/Kaggle secrets or env var, not directly in the notebook.
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
assert GROQ_API_KEY, "Set GROQ_API_KEY as an environment variable before running."

# Strong/cheap options: llama-3.1-8b-instant, llama-3.3-70b-versatile if available on your account
JUDGE_MODEL = "llama-3.1-8b-instant"
DOUBLE_CHECK = True

client = Groq(api_key=GROQ_API_KEY)
print(f"Judge model: {JUDGE_MODEL}")

## Step 2: Load Generated Natural Hallucination Samples

In [ ]:
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    samples = json.load(f)

print(f"Loaded {len(samples)} samples from {INPUT_JSON}")
print("Example keys:", samples[0].keys())
print("Current labels:", pd.Series([s.get("hallucinated") for s in samples]).value_counts(dropna=False).to_dict())

## Step 3: Define Strict LLM Judge

In [ ]:
def context_to_text(context):
    if isinstance(context, list):
        return "\n\n".join(str(c) for c in context)
    return str(context)


def build_judge_prompt(sample):
    context = context_to_text(sample.get("context", ""))
    question = sample.get("question", "")
    gold_answer = sample.get("gold_answer", sample.get("answer", ""))
    generated_answer = sample.get("generated_answer", sample.get("answer", ""))

    return f"""You are a strict factuality judge for a RAG hallucination detection benchmark.

Context:
{context}

Question:
{question}

Gold/reference answer:
{gold_answer}

Generated answer to label:
{generated_answer}

Task:
Label whether the generated answer is fully supported.

Rules:
- Label 0 only if every factual claim in the generated answer is supported by the context and it answers the question.
- Label 1 if the generated answer contains any unsupported claim, invented detail, contradiction, wrong entity/date/number, or fails to answer the question.
- If the answer matches the gold answer but is not supported by the retrieved context, label 1.
- If unsure, label 1.

Return exactly this JSON and nothing else:
{{"label": 0 or 1, "reason": "brief reason"}}
"""


def parse_judge_response(text):
    text = text.strip()
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            return {"label": 1, "reason": f"Unparseable judge response: {text[:120]}"}
        obj = json.loads(match.group(0))

    label = int(obj.get("label", 1))
    if label not in (0, 1):
        label = 1
    reason = str(obj.get("reason", ""))[:500]
    return {"label": label, "reason": reason}


def call_judge(sample, max_retries=3):
    prompt = build_judge_prompt(sample)
    last_error = None

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=120,
            )
            return parse_judge_response(response.choices[0].message.content)
        except Exception as e:
            last_error = e
            time.sleep(2 ** attempt)

    return {"label": 1, "reason": f"Judge API error after retries: {last_error}"}


def label_sample(sample):
    first = call_judge(sample)
    if not DOUBLE_CHECK:
        return first

    second = call_judge(sample)

    # Conservative rule: any judge saying hallucinated wins.
    final_label = 1 if (first["label"] == 1 or second["label"] == 1) else 0
    return {
        "label": final_label,
        "reason": f"Judge1: {first['reason']} | Judge2: {second['reason']}",
        "judge1_label": first["label"],
        "judge2_label": second["label"],
    }


print("Judge functions ready.")

## Step 4: Label All Samples

In [ ]:
if os.path.exists(CHECKPOINT_JSON):
    print(f"Loading checkpoint: {CHECKPOINT_JSON}")
    with open(CHECKPOINT_JSON, "r", encoding="utf-8") as f:
        labeled_samples = json.load(f)
else:
    labeled_samples = samples.copy()

start_idx = 0
for i, sample in enumerate(labeled_samples):
    if sample.get("llm_judge_label") is None:
        start_idx = i
        break
else:
    start_idx = len(labeled_samples)

print(f"Starting from index {start_idx} / {len(labeled_samples)}")

for i in tqdm(range(start_idx, len(labeled_samples))):
    judgment = label_sample(labeled_samples[i])

    labeled_samples[i]["hallucinated"] = int(judgment["label"])
    labeled_samples[i]["llm_judge_label"] = int(judgment["label"])
    labeled_samples[i]["llm_judge_reason"] = judgment.get("reason", "")
    labeled_samples[i]["llm_judge_model"] = JUDGE_MODEL
    if "judge1_label" in judgment:
        labeled_samples[i]["llm_judge1_label"] = int(judgment["judge1_label"])
        labeled_samples[i]["llm_judge2_label"] = int(judgment["judge2_label"])

    if (i + 1) % 20 == 0:
        with open(CHECKPOINT_JSON, "w", encoding="utf-8") as f:
            json.dump(labeled_samples, f, indent=2, ensure_ascii=False)

with open(CHECKPOINT_JSON, "w", encoding="utf-8") as f:
    json.dump(labeled_samples, f, indent=2, ensure_ascii=False)

print("Labeling complete.")

## Step 5: Save Labeled JSON and CSV

In [ ]:
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(labeled_samples, f, indent=2, ensure_ascii=False)

rows = []
for s in labeled_samples:
    rows.append({
        "id": s.get("id"),
        "question": s.get("question"),
        "gold_answer": s.get("gold_answer"),
        "generated_answer": s.get("generated_answer", s.get("answer")),
        "hallucinated": s.get("hallucinated"),
        "llm_judge_reason": s.get("llm_judge_reason"),
        "perplexity": s.get("perplexity"),
        "mean_prob": s.get("mean_prob"),
        "sds": s.get("sds"),
        "context_preview": context_to_text(s.get("context", ""))[:500],
    })

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved labeled JSON: {OUTPUT_JSON}")
print(f"Saved labeled CSV:  {OUTPUT_CSV}")
print("\nLabel distribution:")
print(df["hallucinated"].value_counts().sort_index())
print("\nHallucination rate:", df["hallucinated"].mean())

## Step 6: Evaluate Existing Metrics on LLM Labels

In [ ]:
labels = df["hallucinated"].to_numpy()

if len(np.unique(labels)) < 2:
    print("Only one label class present; AUROC cannot be computed.")
else:
    metric_cols = [c for c in ["perplexity", "mean_prob", "sds"] if c in df.columns and df[c].notna().all()]
    for col in metric_cols:
        scores = df[col].to_numpy(dtype=float)
        if col == "mean_prob":
            scores = -scores
        auroc = roc_auc_score(labels, scores)
        print(f"{col} AUROC: {auroc:.4f}")